In [1]:
pip install numpy pandas matplotlib torch torchvision

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "C:\Users\Scc\AppData\Roaming\Python\Python314\site-packages\torch\lib\torch_cuda.dll" or one of its dependencies.

In [4]:
# --- Task 1: Part A — Handcrafted Filters ---
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Setup device and seeds
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

# Load a single sample from CIFAR-10
temp_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transforms.ToTensor())
img_tensor, label = temp_dataset[0]  # Shape: (3, 32, 32)
img_batch = img_tensor.unsqueeze(0)  # Shape: (1, 3, 32, 32)

# Define filters
v_edge = torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]])
h_edge = v_edge.t()
blur = torch.ones(3, 3) / 9.0

# Initialize convolutional layers and inject weights manually
# 3 input channels (RGB), 1 output channel, 3x3 kernel
conv_v = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)
conv_h = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)
conv_b = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)

# Replicate the 3x3 filter across all 3 input color channels
with torch.no_grad():
    conv_v.weight.copy_(v_edge.repeat(1, 3, 1, 1))
    conv_h.weight.copy_(h_edge.repeat(1, 3, 1, 1))
    conv_b.weight.copy_(blur.repeat(1, 3, 1, 1))

# Process image through filters
with torch.no_grad():
    out_v = conv_v(img_batch).squeeze().numpy()
    out_h = conv_h(img_batch).squeeze().numpy()
    out_b = conv_b(img_batch).squeeze().numpy()

# Convert original tensor back to HWC format for plotting
orig_img = img_tensor.permute(1, 2, 0).numpy()

# Plotting results
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(orig_img)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(out_v, cmap="gray")
axes[1].set_title("Vertical Edges")
axes[1].axis('off')

axes[2].imshow(out_h, cmap="gray")
axes[2].set_title("Horizontal Edges")
axes[2].axis('off')

axes[3].imshow(out_b, cmap="gray")
axes[3].set_title("Blur Filter")
axes[3].axis('off')

plt.tight_layout()
plt.show()# --- Task 1: Part A — Handcrafted Filters ---
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Setup device and seeds
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

# Load a single sample from CIFAR-10
temp_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transforms.ToTensor())
img_tensor, label = temp_dataset[0]  # Shape: (3, 32, 32)
img_batch = img_tensor.unsqueeze(0)  # Shape: (1, 3, 32, 32)

# Define filters
v_edge = torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]])
h_edge = v_edge.t()
blur = torch.ones(3, 3) / 9.0

# Initialize convolutional layers and inject weights manually
# 3 input channels (RGB), 1 output channel, 3x3 kernel
conv_v = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)
conv_h = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)
conv_b = nn.Conv2d(3, 1, kernel_size=3, padding=1, bias=False)

# Replicate the 3x3 filter across all 3 input color channels
with torch.no_grad():
    conv_v.weight.copy_(v_edge.repeat(1, 3, 1, 1))
    conv_h.weight.copy_(h_edge.repeat(1, 3, 1, 1))
    conv_b.weight.copy_(blur.repeat(1, 3, 1, 1))

# Process image through filters
with torch.no_grad():
    out_v = conv_v(img_batch).squeeze().numpy()
    out_h = conv_h(img_batch).squeeze().numpy()
    out_b = conv_b(img_batch).squeeze().numpy()

# Convert original tensor back to HWC format for plotting
orig_img = img_tensor.permute(1, 2, 0).numpy()

# Plotting results
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(orig_img)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(out_v, cmap="gray")
axes[1].set_title("Vertical Edges")
axes[1].axis('off')

axes[2].imshow(out_h, cmap="gray")
axes[2].set_title("Horizontal Edges")
axes[2].axis('off')

axes[3].imshow(out_b, cmap="gray")
axes[3].set_title("Blur Filter")
axes[3].axis('off')

plt.tight_layout()
plt.show()

# --- Task 1: Part B — Shape Tracking ---

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)

    def forward(self, x):
        print(f"Input shape:  {x.shape}")
        x = self.conv1(x)
        print(f"After conv1:  {x.shape}")
        x = self.pool1(x)
        print(f"After pool1:  {x.shape}")
        x = self.conv2(x)
        print(f"After conv2:  {x.shape}")
        x = self.pool2(x)
        print(f"After pool2:  {x.shape}")
        return x

# Execute the tracking test
dummy_input = torch.randn(8, 3, 32, 32)
shape_tracker = TinyCNN()
_ = shape_tracker(dummy_input)

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "C:\Users\Scc\AppData\Roaming\Python\Python314\site-packages\torch\lib\torch_cuda.dll" or one of its dependencies.

In [6]:
# --- Task 2: Standard CIFAR-10 Classification ---
import torch.optim as optim

# Data Preparations
basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

train_set_base = datasets.CIFAR10(root="./data", train=True, download=True, transform=basic_transform)
val_set_base = datasets.CIFAR10(root="./data", train=False, download=True, transform=basic_transform)

train_loader_base = DataLoader(train_set_base, batch_size=128, shuffle=True)
val_loader_base = DataLoader(val_set_base, batch_size=128, shuffle=False)

# Architecture definition
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), # Output: 32 x 16 x 16
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2) # Output: 64 x 8 x 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model_base = CIFAR10CNN().to(device)

# Total Parameters Check
total_params = sum(p.numel() for p in model_base.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,}")

# Reusable Training Routine
def run_training_experiment(model, train_loader, val_loader, epochs=15):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        correct_train = 0
        total_train = 0
        running_loss = 0.0
        
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()
            
        train_loss = running_loss / total_train
        train_acc = 100. * correct_train / total_train
        
        # Validation
        model.eval()
        val_loss_total = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                
                val_loss_total += loss.item() * imgs.size(0)
                _, predicted = outputs.max(1)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()
                
        val_loss = val_loss_total / total_val
        val_acc = 100. * correct_val / total_val
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1:02d} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
        
    return history

# Run Baseline Training
print("--- Starting Baseline Training ---")
history_base = run_training_experiment(model_base, train_loader_base, val_loader_base)

# --- Plotting Baseline Results ---
def plot_curves(history, title_suffix=""):
    epochs_range = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Train Loss')
    plt.plot(epochs_range, history['val_loss'], label='Val Loss')
    plt.title(f'Loss Curves {title_suffix}')
    plt.xlabel('Epoch')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_acc'], label='Train Acc')
    plt.plot(epochs_range, history['val_acc'], label='Val Acc')
    plt.title(f'Accuracy Curves {title_suffix}')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

plot_curves(history_base, "(Baseline)")

OSError: [WinError 193] %1 is not a valid Win32 application. Error loading "C:\Users\Scc\AppData\Roaming\Python\Python314\site-packages\torch\lib\torch_cuda.dll" or one of its dependencies.

In [7]:
# --- Task 3: Augmented Data Training ---

# Richer transforms pipeline
train_tf_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

train_set_aug = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tf_aug)
train_loader_aug = DataLoader(train_set_aug, batch_size=128, shuffle=True)

# Build a separate model instance to run clean from scratch
model_aug = CIFAR10CNN().to(device)

print("--- Starting Augmented Training ---")
history_aug = run_training_experiment(model_aug, train_loader_aug, val_loader_base)
plot_curves(history_aug, "(Augmented)")

# --- Metrics Breakdown and Comparisons ---

best_val_base = max(history_base['val_acc'])
final_train_base = history_base['train_acc'][-1]
gap_base = final_train_base - best_val_base

best_val_aug = max(history_aug['val_acc'])
final_train_aug = history_aug['train_acc'][-1]
gap_aug = final_train_aug - best_val_aug

print("\n--- Summary Performance Table ---")
print(f"{'Run':<30} | {'Best Val Accuracy':<18} | {'Train/Val Gap':<15}")
print("-" * 70)
print(f"{'Task 2 (no augmentation)':<30} | {best_val_base:>17.2f}% | {gap_base:>14.2f}%")
print(f"{'Task 3 (with augmentation)':<30} | {best_val_aug:>17.2f}% | {gap_aug:>14.2f}%")

NameError: name 'transforms' is not defined